In [1]:
import os

%env LOCAL_PROJECT_DIR=local_project_dir

os.environ["LOCAL_DATA_DIR"] = os.path.join(os.getenv("LOCAL_PROJECT_DIR", os.getcwd()), "data")
os.environ["LOCAL_EXPERIMENT_DIR"] = os.path.join(os.getenv("LOCAL_PROJECT_DIR", os.getcwd()), "classification")
os.environ["LOCAL_SPECS_DIR"] = os.path.join(os.getcwd(), "specs")

%env KEY=dHFmaTA4amVydDR0cG8xOTcwM2FoZW5vMmI6ZjZhOTRmYzUtMjkzOS00NTk2LThkNWEtYzk0NDM1Y2EzZDg3
%env NUM_GPUS=1
%env SPECS_DIR=/workspace/tao-experiments/classification/specs
%env USER_EXPERIMENT_DIR=/workspace/tao-experiments/classification
%env DATA_DOWNLOAD_DIR=/workspace/tao-experiments/data

env: LOCAL_PROJECT_DIR=local_project_dir
env: KEY=dHFmaTA4amVydDR0cG8xOTcwM2FoZW5vMmI6ZjZhOTRmYzUtMjkzOS00NTk2LThkNWEtYzk0NDM1Y2EzZDg3
env: NUM_GPUS=1
env: SPECS_DIR=/workspace/tao-experiments/classification/specs
env: USER_EXPERIMENT_DIR=/workspace/tao-experiments/classification
env: DATA_DOWNLOAD_DIR=/workspace/tao-experiments/data


In [3]:
# verify the mounts
!cat ~/.tao_mounts.json

{
    "Mounts": [
        {
            "source": "local_project_dir",
            "destination": "/workspace/tao-experiments"
        },
        {
            "source": "/home/kmilo8346/pactolus_belt_vision/classification/specs",
            "destination": "/workspace/tao-experiments/classification/specs"
        }
    ],
    "DockerOptions": {
        "user": "1001:1002"
    }
}

In [18]:
# filtering > 3 and splitting
import os
import shutil
from glob import glob
from tqdm import tqdm
from random import shuffle

all_eans_dirs = glob(os.path.join(os.getenv('LOCAL_DATA_DIR'), 'training/*'))
all_eans_dirs = [x for x in all_eans_dirs if len(glob(x + '/*')) > 3]
TARGET_DIR = os.path.join(os.getenv("LOCAL_DATA_DIR"), 'fulltrain_training_split')

for dir in tqdm(all_eans_dirs):
  newdir_train = os.path.join(TARGET_DIR, 'train', dir.split('/')[-1])
  newdir_val = os.path.join(TARGET_DIR, 'val', dir.split('/')[-1])
  newdir_test = os.path.join(TARGET_DIR, 'test', dir.split('/')[-1])

  if not os.path.exists(newdir_train):
    os.makedirs(newdir_train)
  if not os.path.exists(newdir_val):
    os.makedirs(newdir_val)
  if not os.path.exists(newdir_test):
    os.makedirs(newdir_test)

  img_list = glob(dir + '/*.jpeg')
  shuffle(img_list)

  for j in range(int(len(img_list)*0.7)):
    shutil.copy2(img_list[j], newdir_train)

  for j in range(int(len(img_list)*0.7), int(len(img_list)*0.8)):
    shutil.copy2(img_list[j], newdir_val)
                
  for j in range(int(len(img_list)*0.8), len(img_list)):
    shutil.copy2(img_list[j], newdir_test)
                
print('Done splitting train dataset.')

100%|██████████| 3149/3149 [08:31<00:00,  6.15it/s]  

Done splitting train dataset.


In [20]:
# count images per split
!find $LOCAL_DATA_DIR/fulltrain_training_split/train -type f | wc -l
!find $LOCAL_DATA_DIR/fulltrain_training_split/val -type f | wc -l
!find $LOCAL_DATA_DIR/fulltrain_training_split/test -type f | wc -l

52195
7798
16521


In [21]:
# View the versions of the TAO launcher
!tao info

Configuration of the TAO Toolkit Instance
dockers: ['nvidia/tao/tao-toolkit-tf', 'nvidia/tao/tao-toolkit-pyt', 'nvidia/tao/tao-toolkit-lm']
format_version: 2.0
toolkit_version: 3.21.11
published_date: 11/08/2021


In [22]:
print("Check that model is downloaded into dir.")
!ls -l $LOCAL_EXPERIMENT_DIR/pretrained_resnet18/pretrained_classification_vresnet18

Check that model is downloaded into dir.
total 91100
-rw------- 1 kmilo8346 kmilo8346 93278448 Feb 24 15:33 resnet_18.hdf5


In [23]:
# checking specs for train
!cat $LOCAL_SPECS_DIR/fulltrain_classification_spec.cfg

model_config {
  arch: "resnet",
  n_layers: 18
  # Setting these parameters to true to match the template downloaded from NGC.
  use_batch_norm: true
  all_projections: true
  freeze_blocks: 0
  freeze_blocks: 1
  input_image_size: "3,224,224"
}
train_config {
  train_dataset_path: "/workspace/tao-experiments/data/stage1_training_split/train"
  val_dataset_path: "/workspace/tao-experiments/data/stage1_training_split/val"
  pretrained_model_path: "/workspace/tao-experiments/classification/pretrained_resnet18/pretrained_classification_vresnet18/resnet_18.hdf5"
  optimizer {
    sgd {
    lr: 0.01
    decay: 0.0
    momentum: 0.9
    nesterov: False
  }
}
  batch_size_per_gpu: 64
  n_epochs: 80
  n_workers: 16
  preprocess_mode: "caffe"
  enable_random_crop: True
  enable_center_crop: True
  label_smoothing: 0.0
  mixup_alpha: 0.1
  # regularizer
  reg_config {
    type: "L2"
    scope: "Conv2D,Dense"
    weight_decay: 0.00005
  }

  # learning_rate
  lr_config {
    step {
      learnin

In [ ]:
# fulltrain train!!
!tao classification train -e $SPECS_DIR/fulltrain_classification_spec.cfg -r $USER_EXPERIMENT_DIR/fulltrain_output -k $KEY

### BEST 59 0.5810135
### LAST 80 0.57653034

In [2]:
!tao classification evaluate -e $SPECS_DIR/fulltrain_classification_spec.cfg -k $KEY

2022-02-26 16:14:44,641 [INFO] root: Registry: ['nvcr.io']
2022-02-26 16:14:44,706 [INFO] tlt.components.instance_handler.local_instance: Running command in container: nvcr.io/nvidia/tao/tao-toolkit-tf:v3.21.11-tf1.15.5-py3
Matplotlib created a temporary config/cache directory at /tmp/matplotlib-a85y44bc because the default path (/.config/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Using TensorFlow backend.
Using TensorFlow backend.

2022-02-26 16:15:03,613 [WARNING] tensorflow: From /root/.cache/bazel/_bazel_root/ed34e6d125608f91724fda23656f1726/execroot/ai_infra/bazel-out/k8-fastbuild/bin/magnet/packages/iva/build_wheel.runfiles/ai_infra/iva/makenet/scripts/evaluate.py:28: The name tf.ConfigProto is deprecated. Please use tf.compat.v1.ConfigProto instead.


2022-02-26 16:15:03,613 [WARNING] tensorflow: Fr

### prune & retrain

In [2]:
%env EPOCH=059
!mkdir -p $LOCAL_EXPERIMENT_DIR/fulltrain_output/resnet_pruned

env: EPOCH=059


In [3]:
!tao classification prune -m $USER_EXPERIMENT_DIR/fulltrain_output/weights/resnet_$EPOCH.tlt \
           -o $USER_EXPERIMENT_DIR/fulltrain_output/resnet_pruned/resnet18_nopool_bn_pruned.tlt \
           -eq union \
           -pth 0.6 \
           -k $KEY

2022-02-26 23:10:28,919 [INFO] root: Registry: ['nvcr.io']
2022-02-26 23:10:28,976 [INFO] tlt.components.instance_handler.local_instance: Running command in container: nvcr.io/nvidia/tao/tao-toolkit-tf:v3.21.11-tf1.15.5-py3
Matplotlib created a temporary config/cache directory at /tmp/matplotlib-ba8x7niu because the default path (/.config/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Using TensorFlow backend.
Using TensorFlow backend.
2022-02-26 23:10:50,267 [INFO] modulus.pruning.pruning: Exploring graph for retainable indices
2022-02-26 23:10:50,853 [INFO] modulus.pruning.pruning: Pruning model and appending pruned nodes to new graph
2022-02-26 23:11:07,186 [INFO] iva.common.magnet_prune: Pruning ratio (pruned model / original model): 0.884138359150141
2022-02-26 23:11:07,979 [INFO] root: Pruning ratio (pru

In [5]:
print('Pruned model:')
print('------------')
!ls -r1t $LOCAL_EXPERIMENT_DIR/fulltrain_output/resnet_pruned

Pruned model:
------------
resnet18_nopool_bn_pruned.tlt


In [8]:
!cat $LOCAL_SPECS_DIR/fulltrain_classification_retrain_spec.cfg

model_config {
  arch: "resnet",
  n_layers: 18
  use_batch_norm: true
  all_projections: true
  input_image_size: "3,224,224"
}
train_config {
  train_dataset_path: "/workspace/tao-experiments/data/fulltrain_training_split/train"
  val_dataset_path: "/workspace/tao-experiments/data/fulltrain_training_split/val"
  pretrained_model_path: "/workspace/tao-experiments/classification/fulltrain_output/resnet_pruned/resnet18_nopool_bn_pruned.tlt"
  optimizer {
    sgd {
    lr: 0.01
    decay: 0.0
    momentum: 0.9
    nesterov: False
  }
}
  batch_size_per_gpu: 64
  n_epochs: 80
  n_workers: 16
  preprocess_mode: "caffe"
  enable_random_crop: True
  enable_center_crop: True
  label_smoothing: 0.0
  mixup_alpha: 0.1
  # regularizer
  reg_config {
    type: "L2"
    scope: "Conv2D,Dense"
    weight_decay: 0.00005
  }

  # learning_rate
  lr_config {
    step {
      learning_rate: 0.006
      step_size: 10
      gamma: 0.1
    }
  }
}
eval_config {
  eval_dataset_path: "/workspace/tao-experime

In [ ]:
# re-train
!tao classification train -e $SPECS_DIR/fulltrain_classification_retrain_spec.cfg \
                      -r $USER_EXPERIMENT_DIR/fulltrain_output_retrain \
                      -k $KEY